<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week1_fundamentals/Week1_Lesson2_Workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠 IB CS — Week 1 · Lesson 2 (Workshop)
## Workshop "Data Integrity" — practice for A4.2.1
### A4.2 Data Preprocessing (HL — Higher Level)

> 📘 Syllabus: **A4.2.1 — *Describe* the significance of data cleaning.**
>
> ⏱ Duration: ~90 minutes (a double lesson).
> 💻 Platform: Google Colab, or local Jupyter.
> 🎯 Goal: work through a **full preprocessing cycle** by hand, so you understand what each step does and why it matters.

---

### ⚠️ Before the workshop starts:

1. Open this notebook in **Google Colab** (`File → Open → GitHub`) or upload it.
2. Run each cell **yourself**; do not only read the code.
3. When you see a `🎯 TRY IT` task, stop and do it.
4. Write down your IB-style answers to marked questions; they will appear in homework.

---

### 📋 Workshop plan:

| Part | Topic | Time |
|---|---|---|
| 1 | Loading and first inspection (EDA) | 15 min |
| 2 | Handling missing values (imputation) | 25 min |
| 3 | Detecting outliers: Z-score and IQR | 30 min |
| 4 | Normalisation vs standardisation | 25 min |
| 5 | Before/after: final visualisation | 15 min |
| 6 | Writing IB-style conclusions | 10 min |


## Part 1 · Loading and first inspection of the data

We will create a synthetic, deliberately messy student dataset — the kind of data you might run into in an IA or a Section B scenario.


In [ ]:
# Import libraries. This is a standard setup for an IB ML task.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_style('whitegrid')

# Create a "dirty" dataset with 200 students
n = 200
data = {
    'student_id':    range(1, n+1),
    'age':           np.random.normal(17, 1.5, n).round(0),
    'hours_study':   np.random.normal(15, 5, n).round(1),
    'attendance_%':  np.random.normal(85, 10, n).round(1),
    'sleep_hours':   np.random.normal(7, 1.2, n).round(1),
    'final_score':   np.random.normal(75, 12, n).round(0)
}
df = pd.DataFrame(data)

# === Deliberately make the data dirty ===
# 1. Missing values (NaN)
df.loc[np.random.choice(df.index, 15, replace=False), 'hours_study'] = np.nan
df.loc[np.random.choice(df.index, 10, replace=False), 'sleep_hours'] = np.nan
df.loc[np.random.choice(df.index, 8,  replace=False), 'attendance_%'] = np.nan

# 2. Outliers: unrealistic values
df.loc[5, 'hours_study']  = 80      # 80 hours per week is unrealistic
df.loc[12, 'sleep_hours'] = 0.5     # 0.5 hours of sleep
df.loc[25, 'final_score'] = 150     # max should be 100
df.loc[40, 'age']         = 50      # teacher?
df.loc[77, 'attendance_%'] = -10    # negative attendance

# 3. Duplicates
df = pd.concat([df, df.iloc[[3, 7]]], ignore_index=True)

print(f"Dataset shape: {df.shape}")
df.head(10)


### 🔍 EDA — Exploratory Data Analysis

> 💎 **Secret #1:** in the IB IA, examiners expect EDA **at the beginning**. It shows that you understand the data before building a model.


In [ ]:
# 1. Basic information
print("=== df.info() ===")
df.info()
print("\n=== Missing values by column ===")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")
print(f"Duplicates: {df.duplicated().sum()}")


In [ ]:
# 2. Descriptive statistics: a key tool in the IB IA
df.describe()


> 🎯 **TRY IT #1:** Look at `describe()`. What looks **suspicious**?
>
> Hints:
> - `age` — what is the maximum value?
> - `attendance_%` — what is the minimum?
> - `hours_study` — what is the maximum?
> - `final_score` — what is the maximum?
>
> 📝 Write down 3-4 anomalies. These are exactly what we need to clean.


In [ ]:
# 3. Visual inspection: boxplots for all numeric columns
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, ['age', 'hours_study', 'attendance_%', 'sleep_hours', 'final_score']):
    sns.boxplot(y=df[col], ax=ax, color='lightblue')
    ax.set_title(col, fontsize=11, fontweight='bold')
plt.suptitle('Boxplots BEFORE cleaning — points beyond the whiskers = potential outliers', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()


### 📚 IB definitions (from Baumgarten, A4.2.1)

> **Outlier:** *a data point that deviates from the typical pattern of values in a data set, indicating a possible unusual or erroneous value that should be discounted.*

**7 data cleaning steps from Baumgarten:**

1. **Handling outliers** — IQR, Z-score, capping, removal
2. **Removing duplicate data** — remove repeated rows
3. **Identifying incorrect data** — validation, such as impossible negative attendance
4. **Filtering irrelevant data** — remove irrelevant columns
5. **Transform improperly formatted data** — standardise formats
6. **Missing data** — fill missing values (imputation)
7. **Normalization and standardization** — scaling

Today we practise steps **1, 2, 3, 6, and 7**.


## Part 2 · Handling missing values (Imputation)

> 📘 **IB Top Tip from Baumgarten:**
> *"Inputting missing data will improve model accuracy through providing a complete data set, but it can introduce bias if the resulting data set does not match actual data distribution."*

**3 main strategies:**

| Method | When to use it | Drawback |
|---|---|---|
| **Deletion** (remove rows with NaN) | very few missing values (<5%) | loses data |
| **Mean / Median imputation** | missing values are random (MCAR) | changes distribution and reduces variance |
| **Mode imputation** | categorical variables | same issue |
| **KNN / Regression imputation** | many missing values, higher accuracy needed | more complex and slower |


In [ ]:
# Strategy A: inspect the number of missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'Missing values': missing, '%': missing_pct})


In [ ]:
# Strategy B: mean imputation for hours_study
# Keep a copy before cleaning for comparison
df_before = df.copy()

# Fill missing values with the mean
mean_hours = df['hours_study'].mean()
df['hours_study'] = df['hours_study'].fillna(mean_hours)
print(f"Filled with mean: {mean_hours:.2f} hours")

# Median imputation for sleep_hours: more robust to outliers
median_sleep = df['sleep_hours'].median()
df['sleep_hours'] = df['sleep_hours'].fillna(median_sleep)
print(f"Filled with median: {median_sleep:.2f} hours")

# For attendance_%, use the median because there is an outlier at -10
median_att = df['attendance_%'].median()
df['attendance_%'] = df['attendance_%'].fillna(median_att)
print(f"Filled with median: {median_att:.2f}%")

print("\n=== Missing values AFTER ===")
print(df.isnull().sum())


### 🎯 TRY IT #2: Why did I choose **mean** for `hours_study`, but **median** for `sleep_hours`?

> 💡 Think about the difference:
> - **Mean** is sensitive to outliers.
> - **Median** is robust to outliers.
>
> Which anomalous value exists in `sleep_hours`? Go back to the code that made the data dirty. That is why median is safer.

> 💎 **IB secret #2:** in the exam, always **justify** your method. "I used mean" earns little. *"I used median because the data contains outliers (e.g., sleep_hours = 0.5), which would skew the mean"* earns full marks.


In [ ]:
# Visualisation: distribution BEFORE and AFTER imputation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_before['hours_study'].dropna(), bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(mean_hours, color='red', linestyle='--', label=f'Mean = {mean_hours:.1f}')
axes[0].set_title('hours_study — BEFORE (with missing values)')
axes[0].set_xlabel('Hours'); axes[0].legend()

axes[1].hist(df['hours_study'], bins=20, color='green', alpha=0.7, edgecolor='black')
axes[1].axvline(mean_hours, color='red', linestyle='--', label=f'Mean = {mean_hours:.1f}')
axes[1].set_title('hours_study — AFTER mean imputation')
axes[1].set_xlabel('Hours'); axes[1].legend()

plt.tight_layout(); plt.show()


> 🔬 **Observation:** notice that the peak around the mean became higher. This is **bias**: mean imputation can **distort** the distribution by artificially increasing density near the mean.
>
> This is exactly what Baumgarten warns about. In the exam, you can use it as an argument in `Discuss` questions.


In [ ]:
# Part 2.5 · Removing duplicates
print(f"Duplicates before: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicates after: {df.duplicated().sum()}")
print(f"Dataset shape: {df.shape}")


## Part 3 · Detecting outliers: Z-score and IQR

> **Outlier** (syllabus): a data point that deviates from the typical pattern of values...
>
> Two main methods from Baumgarten, p. 22:

### Method 1: **Z-score** (standardised deviation)

$$Z = \frac{x - \mu}{\sigma}$$

- $\mu$ is the mean, $\sigma$ is the standard deviation.
- **Rule:** if $|Z| > 3$, the value is an outlier under the 3-sigma rule for a normal distribution.

### Method 2: **IQR** (Interquartile Range) — MORE ROBUST to outliers

$$IQR = Q_3 - Q_1$$

- A value is an outlier if $x < Q_1 - 1.5 \cdot IQR$ or $x > Q_3 + 1.5 \cdot IQR$.

> 💎 **Secret #3:** Z-score assumes a **normal** distribution. If the data is skewed, use **IQR**. In the exam, saying this can earn depth marks.


In [ ]:
# === Z-score method ===
def find_outliers_zscore(series, threshold=3):
    z = np.abs(stats.zscore(series))
    return series[z > threshold]

# === IQR method ===
def find_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series[(series < lower) | (series > upper)], lower, upper

# Check final_score: it contains 150, which is impossible for a 0-100 score
print("=== final_score ===")
print("\nZ-score outliers:")
print(find_outliers_zscore(df['final_score']))

outliers, low, high = find_outliers_iqr(df['final_score'])
print(f"\nIQR outliers (valid range: {low:.1f} — {high:.1f}):")
print(outliers)


In [ ]:
# Visualise outliers across all columns
numeric_cols = ['age', 'hours_study', 'attendance_%', 'sleep_hours', 'final_score']
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, col in enumerate(numeric_cols):
    # Top: boxplot
    sns.boxplot(y=df[col], ax=axes[0, i], color='salmon')
    axes[0, i].set_title(f'{col}\n(Boxplot — IQR)', fontsize=11)

    # Bottom: histogram with Z-score boundaries
    axes[1, i].hist(df[col], bins=20, color='lightblue', edgecolor='black')
    mu, sigma = df[col].mean(), df[col].std()
    axes[1, i].axvline(mu - 3*sigma, color='red', linestyle='--', label='−3σ')
    axes[1, i].axvline(mu + 3*sigma, color='red', linestyle='--', label='+3σ')
    axes[1, i].axvline(mu, color='green', label='mean')
    axes[1, i].set_title(f'{col}\n(Histogram + Z-score)', fontsize=11)
    axes[1, i].legend(fontsize=8)

plt.tight_layout(); plt.show()


### 🎯 TRY IT #3: What should we do with outliers?

Baumgarten gives **three options**:
1. **Capping**: limit to a boundary, e.g. `final_score = 150 → 100`
2. **Transformation**: log, square root, and so on
3. **Removal**: delete the record

> 💡 **The choice depends on context:**
> - `final_score = 150` is clearly an **input error** because the maximum is 100. → **Cap** at 100, or remove.
> - `attendance_% = −10` is impossible. → **Remove** or replace with the median.
> - `age = 50` may be valid in some contexts, but in a school dataset it is probably an error. **Context matters.**

> 💎 **Secret #4:** for the exam question *"Describe how outliers could affect the performance of the linear regression model"*, a strong answer is: *outliers disproportionately influence the slope of the regression line because least-squares minimization is sensitive to large residuals; this can lead to poor predictions for typical data points.*


In [ ]:
# Apply capping for final_score and attendance
df['final_score'] = df['final_score'].clip(0, 100)
df['attendance_%'] = df['attendance_%'].clip(0, 100)

# For age, remove clear errors: assume school age should be at most 19
df = df[(df['age'] >= 14) & (df['age'] <= 19)].reset_index(drop=True)

# For sleep_hours = 0.5, cap the minimum at 3 because less is unrealistic here
df['sleep_hours'] = df['sleep_hours'].clip(3, 12)

# For hours_study = 80, cap the maximum at 60
df['hours_study'] = df['hours_study'].clip(0, 60)

print("After outlier handling:")
df.describe()


## Part 4 · Normalisation vs Standardisation (A4.2.1, final part)

> 📘 **Baumgarten:**
> *"Normalization can be used to rescale input data to a range of [0,1] or [−1,1]. Standardization can be used to transform the input data to have a mean of 0 and standard deviation of 1 (Gaussian distribution)."*

### 🔑 Difference (often confused):

| | **Normalization (Min-Max)** | **Standardization (Z-score)** |
|---|---|---|
| Formula | $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$ | $x' = \frac{x - \mu}{\sigma}$ |
| Range | [0, 1] | mean=0, std=1, no fixed range |
| When to use | KNN, neural networks, images | algorithms that benefit from normal-like scaling, such as linear/logistic regression |
| Sensitivity to outliers | **high** because an outlier stretches the scale | medium |

> 🚨 **Common mistake (Baumgarten, direct quote):**
> *"Ignoring the important role that normalization and standardization play. Apply these transformations consistently across all data used in the model."*

> 💎 **Secret #5:** for *"Why do KNN algorithms need normalization?"*, answer: KNN calculates **distances**. If one feature is `salary` with range 0-1,000,000 and another is `age` with range 0-100, salary will dominate the distance calculation. Normalisation makes their contributions comparable.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

cols_to_scale = ['hours_study', 'attendance_%', 'sleep_hours', 'final_score']

# Normalisation (Min-Max)
mm = MinMaxScaler()
df_norm = df.copy()
df_norm[cols_to_scale] = mm.fit_transform(df[cols_to_scale])

# Standardisation (Z-score)
ss = StandardScaler()
df_std = df.copy()
df_std[cols_to_scale] = ss.fit_transform(df[cols_to_scale])

print("=== ORIGINAL ===")
print(df[cols_to_scale].describe().loc[['mean', 'std', 'min', 'max']].round(2))
print("\n=== NORMALIZED (Min-Max, should be [0,1]) ===")
print(df_norm[cols_to_scale].describe().loc[['mean', 'std', 'min', 'max']].round(2))
print("\n=== STANDARDIZED (mean=0, std=1) ===")
print(df_std[cols_to_scale].describe().loc[['mean', 'std', 'min', 'max']].round(2))


In [ ]:
# Visualisation: the same column in 3 formats
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['final_score'], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Original\n(0-100)', fontweight='bold')
axes[0].set_xlabel('final_score')

axes[1].hist(df_norm['final_score'], bins=20, color='green', edgecolor='black')
axes[1].set_title('Normalized\n[0, 1]', fontweight='bold')
axes[1].set_xlabel('final_score (norm)')

axes[2].hist(df_std['final_score'], bins=20, color='orange', edgecolor='black')
axes[2].set_title('Standardized\n(mean=0, std=1)', fontweight='bold')
axes[2].set_xlabel('final_score (std)')

plt.tight_layout(); plt.show()


## Part 5 · Final comparison: BEFORE ↔ AFTER

Now we show **exactly what improved** because of cleaning.


In [ ]:
# Comparative boxplots
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, col in enumerate(['age', 'hours_study', 'attendance_%', 'sleep_hours', 'final_score']):
    sns.boxplot(y=df_before[col].dropna(), ax=axes[0, i], color='salmon')
    axes[0, i].set_title(f'BEFORE — {col}', fontweight='bold')

    sns.boxplot(y=df[col], ax=axes[1, i], color='lightgreen')
    axes[1, i].set_title(f'AFTER — {col}', fontweight='bold')

plt.suptitle('Comparison BEFORE ↔ AFTER cleaning', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Correlation matrix: after cleaning, the signal should be easier to read
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df_before[['age','hours_study','attendance_%','sleep_hours','final_score']].corr(),
            annot=True, cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Correlations BEFORE cleaning')

sns.heatmap(df[['age','hours_study','attendance_%','sleep_hours','final_score']].corr(),
            annot=True, cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Correlations AFTER cleaning')

plt.tight_layout(); plt.show()


### 🎯 TRY IT #4: Compare correlations "Before" and "After".

Did the values change? Which feature now correlates most strongly with `final_score`?

> 💡 This is **critical** for the next stage: **feature selection** (A4.2.2). With dirty data, correlations lie.


## Part 6 · Writing IB-style conclusions

> 💎 **Secret #6:** in the IB IA / Section B, you need to **justify every preprocessing step**. Use this structure:
>
> **Step → Method → Justification → Effect on the model**

### Model IB-style conclusion (adapt it for your report):

```
1. Missing values:
   - In the `hours_study` column (15 missing values, 7.5%), mean imputation was used
     because the distribution is close to normal and the missing values appear random.
   - In the `sleep_hours` column (10 missing values), MEDIAN imputation was used
     because an outlier (0.5) would skew the mean.

2. Outliers:
   - In `final_score`, capping at 100 was used because the value 150 is impossible
     when the maximum score is 100.
   - In `attendance_%`, the negative value was corrected using capping to [0, 100].

3. Normalization:
   - MinMaxScaler was applied to all numeric features before using KNN,
     because KNN is based on Euclidean distance and is sensitive to different scales.

Effect on model:
- Cleaning reduced standard deviation of `final_score` from 13.2 to 11.8,
  removing artificial inflation caused by the outlier.
- Normalization ensured all features contribute equally to distance calculations,
  preventing dominance of `attendance_%` (range 0-100) over `sleep_hours` (range 3-12).
```


## ✅ What we did in the workshop

- [x] Loaded a "dirty" dataset and performed EDA
- [x] Handled **missing values** (mean / median imputation) — A4.2.1 ✓
- [x] Found **outliers** using two methods (Z-score, IQR) and handled them (capping, removal) — A4.2.1 ✓
- [x] Removed **duplicates** — A4.2.1 ✓
- [x] Applied **normalisation** and **standardisation** — A4.2.1 ✓
- [x] Visualised BEFORE ↔ AFTER — useful IA technique
- [x] Learned how to write an **IB-style justification** for each step

---

### 🏠 Homework (see `Week1_HW2_Practice.ipynb`)

> A real "dirty" house-price dataset. Full preprocessing cycle + IB-style report.

---

### 💎 Final workshop secrets

1. **Always** justify your method choice; this gives about half the marks.
2. **Always** show BEFORE ↔ AFTER; plots give visual evidence.
3. **Cap before normalising**: outliers distort min-max scaling.
4. **Median > Mean** when outliers are present.
5. For *"Describe the significance of data cleaning"* (a typical 4-mark question), use this structure:
   - What it improves: accuracy, generalisation
   - What it prevents: bias, overfitting
   - A concrete example
   - A trade-off: data loss when removing records
